# FAQ And Troubleshooting


## My TMAP has detached branches, or TMAP warns about disconnected components

With the default `connect_components=True`, TMAP returns a single tree. If the kNN graph splits into several pieces (usually because `n_neighbors` is too low), TMAP adds the lowest-weight edges that join the pieces and warns. `model.n_components_` is the number of pieces it found and `model.n_bridges_` is the number of edges it added. Pass `connect_components=False` to keep the separate trees.

If you see this warning, or branches that look detached, try these in order:

1. increase `n_neighbors` (all metrics)
2. for sets, strings or sparse input (the MinHash + LSHForest path), increase `kc` and `n_permutations`
3. check whether your data really is split into separate groups


## My map changes between runs

The layout step is always seeded and deterministic: the same kNN graph and the same `seed` give identical coordinates. If you pass a `LayoutConfig`, the estimator sets its `deterministic` and `seed` fields itself, on the object you pass.

Differences between runs come from the kNN step, which depends on the backend:

- **MinHash + LSHForest** (sets, strings, sparse matrices): deterministic for a given `minhash_seed`.
- **USearch exact search** (cosine and euclidean below 50,000 points): deterministic.
- **USearch HNSW** (dense binary matrices, and cosine or euclidean at 50,000 points and above): the index is built on several threads, so repeated runs can return slightly different neighbors. That can change the tree and the layout.

For identical coordinates on every run, pass `reproducible=True`. TMAP then builds the HNSW index on one thread, which is slower (roughly 5-7x for the HNSW build at 10,000+ points).

```python
model = TMAP(metric="jaccard", reproducible=True).fit(binary_matrix)
```

## TMAP is slow

- for sets, strings or sparse input (the MinHash + LSHForest path), try smaller `n_permutations`
- for dense metrics, USearch uses exact search for small datasets and HNSW for larger ones
- reduce `layout_iterations` if layout quality is good enough
- `reproducible=True` makes the HNSW build several times slower; leave it off if you do not need identical runs


## Do cosine and euclidean need extra installation?

No. Dense metrics use USearch from the core install:

```
pip install tmap2
```

PyPI has wheels for Linux x86_64, macOS arm64 and Windows x86_64 on Python 3.11 to 3.13. Building from source (for example `pip install .` in a clone) needs CMake and a C++17 compiler. Optional extras add domain tools: `tmap2[chemistry]` (RDKit), `tmap2[notebook]` (jupyter-scatter), `tmap2[proteins]` (Biopython), `tmap2[singlecell]` (anndata), or `tmap2[all]`.


## Why do edges not show in the notebook widget, and why does it warn?

Notebook widgets focus on points, colors, and filters. When a `TmapViz` that holds tree edges is shown as a widget, it warns "Edges are not supported in notebook mode yet" and skips them. Build the viewer with `model.to_tmapviz(include_edges=False)` when you only need the widget. Use HTML export when you want browser rendering with tree edges.


## How do I choose between Jaccard and cosine/euclidean?

- **Jaccard** (default): binary fingerprints, one-hot encodings, sparse binary data. A dense 0/1 matrix (NumPy array or DataFrame) uses USearch. Lists of feature indices, lists of strings and SciPy sparse matrices use MinHash + LSHForest (see the last question below).
- **Cosine**: dense embeddings (ESM-2, ProtTrans, sentence transformers). Uses USearch.
- **Euclidean**: dense features where absolute magnitude matters. Uses USearch.
- **Precomputed**: you already have a distance matrix. Pass it directly.

```python
# Binary fingerprints → jaccard
TMAP(metric="jaccard").fit(binary_matrix)

# Dense embeddings → cosine or euclidean
TMAP(metric="cosine").fit(embeddings)

# Pre-computed distances
TMAP(metric="precomputed").fit(distance_matrix)
```

## How do I add new points to an existing map?

`add_points()` inserts new points into a fitted model. It extends the tree and embedding without refitting.
`transform()` places new points on the map without modifying the model.

For cosine and euclidean, fit with `store_index=True`, because both methods query the stored index. On the Jaccard path, `add_points()` needs a binary matrix and raises `TypeError` if `fit()` used sets or strings.

```python
model = TMAP(metric="cosine", store_index=True).fit(X_train)
model.add_points(X_new)       # mutates model, extends tree
coords = model.transform(X_test)  # read-only projection
```

## How do I find paths or distances in the tree?

```python
path = model.path(idx_a, idx_b)        # list of node indices along tree path
d = model.distance(idx_a, idx_b)       # sum of edge weights
pseudotime = model.distances_from(idx) # tree distance to all other points
```

## How do I load protein sequences?

```python
from tmap.utils import read_fasta, read_protein_csv, read_id_list

ids, seqs = read_fasta("proteins.fa")
ids, seqs = read_protein_csv("data.csv", id_col="accession", seq_col="sequence")
ids = read_id_list("uniprot_ids.txt")
```

## How do I save and load a fitted model?

```python
model.save("my_model.pkl")
model = TMAP.load("my_model.pkl")
```

## What is `kc` and when should I change it?

`kc` is the query multiplier for the LSHForest. Each query collects `n_neighbors * kc` candidates from the forest and ranks them by the Jaccard distance estimated from their full MinHash signatures. Higher `kc` usually gives better recall and a better-connected kNN graph, at the cost of longer query time. The default is 50.

`kc` only matters on the MinHash + LSHForest path: sets, strings and sparse input with `metric="jaccard"`. Dense binary matrices and the cosine and euclidean metrics use USearch and ignore it.

## How do I export to HTML for sharing?

```python
# Quick export
model.to_html("map.html")

# Full control via TmapViz
viz = model.to_tmapviz()
viz.title = "My Map"
viz.add_color_layout("Label", labels, categorical=True)
viz.write_html("map.html")
```

## Binary fingerprints: MinHash or USearch Jaccard?

The estimator auto-routes dense binary matrices to USearch, and sets, strings and sparse matrices to MinHash + LSHForest.

| | USearch binary Jaccard | MinHash + LSHForest |
|---|---|---|
| **Distances** | Exact Jaccard | Approximate (via signatures) |
| **Speed** | Fast (multi-threaded HNSW) | Moderate (Numba-accelerated) |
| **Determinism** | Can vary slightly between runs (multi-threaded HNSW build); identical with `reproducible=True` | Fully deterministic for a given `minhash_seed` |

For identical results across runs on the USearch path, pass `reproducible=True` (see "My map changes between runs"). To use the MinHash + LSHForest path, pass lists of feature indices (or a SciPy sparse matrix):

```python
# USearch path (fast)
model = TMAP(metric="jaccard").fit(binary_matrix)

# USearch path, identical results on every run (slower build)
model = TMAP(metric="jaccard", reproducible=True).fit(binary_matrix)

# MinHash + LSH path
sets = [np.where(row)[0].tolist() for row in binary_matrix]
model = TMAP(metric="jaccard", seed=42).fit(sets)
```